# European Macro & Market Intelligence Dashboard

**Data sources:** ECB Statistical Data Warehouse · FRED (St. Louis Fed) · Yahoo Finance  
**Coverage:** 2019 – present | Monthly frequency

This notebook aggregates and visualises key European macroeconomic and market indicators relevant to private equity and investment fund analysis:

| Section | Indicators |
|---|---|
| 1. Monetary Policy | ECB Deposit Rate, Euribor 3M |
| 2. Inflation | Eurozone HICP YoY |
| 3. Equity Markets | EURO STOXX 50, DAX, CAC 40, FTSE MIB, IBEX 35 |
| 4. Energy Prices | EU Natural Gas (TTF), Brent Crude |
| 5. Correlation Analysis | Cross-asset correlation matrix |
| 6. Macro Regime | Rate cycle overlay on equity performance |

---
*When run with live internet access, data is fetched directly from ECB API, FRED, and Yahoo Finance.  
In offline/demo mode, the fetchers automatically fall back to realistic simulated series.*

In [ ]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path

warnings.filterwarnings('ignore')

# Add fetchers to path
sys.path.insert(0, str(Path('.').resolve()))
from fetchers.ecb_fetcher    import get_all_ecb_series
from fetchers.market_fetcher import get_equity_indices, get_energy_prices

OUTPUT_DIR = Path('data/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Chart style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        130,
    'font.family':       'DejaVu Sans',
    'font.size':         9.5,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'axes.prop_cycle':   plt.cycler(color=[
        '#1B4F72', '#2E86C1', '#A93226', '#1E8449',
        '#B7950B', '#7D3C98', '#117A65'
    ]),
})

START = '2019-01'
print('✅ Setup complete. Fetching data...')

In [ ]:
# ── Load all series ───────────────────────────────────────────────────────────
ecb_df    = get_all_ecb_series(START)
equity_df = get_equity_indices(START)
energy_df = get_energy_prices(START)

# Save raw data
ecb_df.to_csv(OUTPUT_DIR / 'ecb_rates_inflation.csv')
equity_df.to_csv(OUTPUT_DIR / 'equity_indices.csv')
energy_df.to_csv(OUTPUT_DIR / 'energy_prices.csv')

print(f'ECB series:    {ecb_df.shape[0]} months, {ecb_df.shape[1]} indicators')
print(f'Equity series: {equity_df.shape[0]} months, {equity_df.shape[1]} indices')
print(f'Energy series: {energy_df.shape[0]} months, {energy_df.shape[1]} commodities')
print(f'\nDate range: {ecb_df.index.min().date()} → {ecb_df.index.max().date()}')

---
## 1. ECB Monetary Policy: Rate Cycle

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

col_dfr     = 'ECB Deposit Rate (%)'
col_euribor = 'Euribor 3M (%)'

ax.fill_between(ecb_df.index, ecb_df[col_dfr], alpha=0.15, color='#1B4F72')
ax.plot(ecb_df.index, ecb_df[col_dfr],     lw=2.0, color='#1B4F72', label='ECB Deposit Rate')
ax.plot(ecb_df.index, ecb_df[col_euribor], lw=1.6, color='#A93226',
        linestyle='--', label='Euribor 3M')

# Annotate key regime changes
annotations = [
    ('2020-03', 'COVID\nResponse', '#555'),
    ('2022-07', 'Hiking\nCycle', '#A93226'),
    ('2024-06', 'First\nCut',    '#1E8449'),
]
for date_str, label, color in annotations:
    d = pd.Timestamp(date_str)
    if d in ecb_df.index or ecb_df.index.min() <= d <= ecb_df.index.max():
        ax.axvline(d, color=color, linewidth=1.0, linestyle=':')
        ax.text(d, ax.get_ylim()[1] * 0.85, label,
                fontsize=7.5, color=color, ha='center',
                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

ax.axhline(0, color='black', linewidth=0.6, linestyle='-')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.set_title('ECB Monetary Policy — Deposit Rate & Euribor 3M', fontweight='bold', pad=10)
ax.set_ylabel('Rate (%)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_ecb_rates.png', bbox_inches='tight')
plt.show()

---
## 2. Eurozone Inflation

In [ ]:
col_cpi = 'Eurozone Inflation YoY (%)'

fig, ax = plt.subplots(figsize=(12, 4))

# Colour bars: red above 2% ECB target, blue below
colors = ['#A93226' if v > 2 else '#2E86C1' for v in ecb_df[col_cpi]]
ax.bar(ecb_df.index, ecb_df[col_cpi], width=25, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(2.0, color='black', linewidth=1.0, linestyle='--', label='ECB Target (2%)')

above = mpatches.Patch(color='#A93226', alpha=0.8, label='Above target')
below = mpatches.Patch(color='#2E86C1', alpha=0.8, label='At or below target')
ax.legend(handles=[above, below, plt.Line2D([0],[0], color='black',
          linestyle='--', label='ECB Target (2%)')], loc='upper left')

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=1))
ax.set_title('Eurozone HICP Inflation YoY — vs ECB 2% Target', fontweight='bold', pad=10)
ax.set_ylabel('Inflation YoY (%)')

peak = ecb_df[col_cpi].max()
peak_date = ecb_df[col_cpi].idxmax()
ax.annotate(f'Peak: {peak:.1f}%',
            xy=(peak_date, peak),
            xytext=(peak_date + pd.DateOffset(months=3), peak - 1.5),
            arrowprops=dict(arrowstyle='->', color='#A93226'),
            fontsize=8.5, color='#A93226')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_inflation.png', bbox_inches='tight')
plt.show()

---
## 3. European Equity Indices — Indexed Performance

In [ ]:
# Rebase all indices to 100 at start
equity_rebased = equity_df.div(equity_df.iloc[0]) * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [3, 1]})

# Top panel: rebased performance
for col in equity_rebased.columns:
    axes[0].plot(equity_rebased.index, equity_rebased[col], lw=1.6, label=col)

axes[0].axhline(100, color='black', linewidth=0.7, linestyle=':')
axes[0].set_title('European Equity Indices — Rebased to 100 (Jan 2019)', fontweight='bold', pad=10)
axes[0].set_ylabel('Index Level (rebased)')
axes[0].legend(loc='upper left', fontsize=8)

# Bottom panel: rolling 3M return of EURO STOXX 50
stoxx = equity_rebased.iloc[:, 0]
roll_ret = stoxx.pct_change(3) * 100
colors = ['#1E8449' if v >= 0 else '#A93226' for v in roll_ret.fillna(0)]
axes[1].bar(roll_ret.index, roll_ret, width=25, color=colors, alpha=0.75)
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].set_ylabel('3M Return (%)')
axes[1].set_title(f'{equity_df.columns[0]} — Rolling 3-Month Return', fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_equity_indices.png', bbox_inches='tight')
plt.show()

# Total return table
total_ret = ((equity_df.iloc[-1] / equity_df.iloc[0]) - 1) * 100
print('\n--- Total Return since Jan 2019 ---')
for idx, val in total_ret.items():
    print(f'  {idx:<18}  {val:+.1f}%')

---
## 4. Energy Prices — European Gas & Brent Crude

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

col_gas   = energy_df.columns[0]
col_brent = energy_df.columns[1]

l1 = ax1.fill_between(energy_df.index, energy_df[col_gas],
                       alpha=0.3, color='#E67E22')
l2, = ax1.plot(energy_df.index, energy_df[col_gas],
               color='#E67E22', lw=2.0, label=col_gas)
l3, = ax2.plot(energy_df.index, energy_df[col_brent],
               color='#1B4F72', lw=1.8, linestyle='--', label=col_brent)

ax1.set_ylabel(col_gas,   color='#E67E22')
ax2.set_ylabel(col_brent, color='#1B4F72')
ax1.tick_params(axis='y', labelcolor='#E67E22')
ax2.tick_params(axis='y', labelcolor='#1B4F72')

# Annotate energy crisis
ax1.axvspan(pd.Timestamp('2021-09'), pd.Timestamp('2022-12'),
            alpha=0.08, color='red', label='Energy Crisis 2021–22')

lines = [l2, l3]
labels = [l.get_label() for l in lines]
labels.append('Energy Crisis 2021–22')
lines.append(mpatches.Patch(alpha=0.15, color='red'))
ax1.legend(lines, labels, loc='upper left', fontsize=8)

ax1.set_title('European Energy Prices — Natural Gas (TTF) & Brent Crude', fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig4_energy_prices.png', bbox_inches='tight')
plt.show()

---
## 5. Cross-Asset Correlation Matrix

In [ ]:
# Build monthly returns for all series
all_series = pd.concat([
    ecb_df[['ECB Deposit Rate (%)', 'Eurozone Inflation YoY (%)']],
    equity_df,
    energy_df
], axis=1).dropna()

returns = all_series.pct_change().dropna()
# For rate/inflation, use level change instead of pct
returns[['ECB Deposit Rate (%)', 'Eurozone Inflation YoY (%)']] = \
    all_series[['ECB Deposit Rate (%)', 'Eurozone Inflation YoY (%)']].diff().dropna()

corr = returns.corr().round(2)

# Shorten labels for display
short_labels = [
    'ECB Rate', 'Inflation',
    'STOXX50', 'DAX', 'CAC40', 'FTSE MIB', 'IBEX35',
    'Gas TTF', 'Brent'
]
corr.index   = short_labels[:len(corr)]
corr.columns = short_labels[:len(corr)]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=35, ha='right', fontsize=8.5)
ax.set_yticklabels(corr.index, fontsize=8.5)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        val = corr.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8,
                color='white' if abs(val) > 0.65 else 'black')

plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title('Cross-Asset Correlation Matrix (Monthly Changes)', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig5_correlation_matrix.png', bbox_inches='tight')
plt.show()

# Save correlation matrix
corr.to_csv(OUTPUT_DIR / 'correlation_matrix.csv')
print('[✓] Correlation matrix saved.')

---
## 6. Macro Regime Analysis — Rate Cycle × Equity Performance

In [ ]:
# Classify macro regime based on rate direction
rate_col   = 'ECB Deposit Rate (%)'
rate_delta = ecb_df[rate_col].diff()

def classify_regime(delta):
    if delta > 0.05:   return 'Hiking'
    elif delta < -0.05: return 'Cutting'
    else:               return 'Stable'

regimes = rate_delta.apply(classify_regime)

# Align equity with regime dates
stoxx_full = equity_df.iloc[:, 0].reindex(ecb_df.index, method='ffill')
stoxx_ret  = stoxx_full.pct_change() * 100

regime_returns = pd.DataFrame({
    'Monthly Return (%)': stoxx_ret,
    'Regime': regimes
}).dropna()

# Summary stats by regime
summary = regime_returns.groupby('Regime')['Monthly Return (%)'].agg(
    Count='count',
    Mean=lambda x: round(x.mean(), 2),
    Median=lambda x: round(x.median(), 2),
    Std=lambda x: round(x.std(), 2),
    Hit_Rate=lambda x: round((x > 0).mean() * 100, 1)
).reset_index()
summary.columns = ['Regime', 'N Months', 'Avg Return (%)', 'Median Return (%)', 'Volatility (%)', 'Hit Rate (%)']

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter regime overlay on STOXX50
palette = {'Hiking': '#A93226', 'Cutting': '#1E8449', 'Stable': '#2E86C1'}
for regime, grp in regime_returns.groupby('Regime'):
    axes[0].scatter(grp.index, grp['Monthly Return (%)'],
                    c=palette[regime], label=regime, s=18, alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.7)
axes[0].set_title('EURO STOXX 50 Monthly Returns by Rate Regime', fontweight='bold')
axes[0].set_ylabel('Monthly Return (%)')
axes[0].legend(title='ECB Regime')

# Right: average return by regime (bar)
bar_colors = [palette[r] for r in summary['Regime']]
bars = axes[1].bar(summary['Regime'], summary['Avg Return (%)'],
                   color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.7)
axes[1].set_title('Average Monthly Return by Rate Regime', fontweight='bold')
axes[1].set_ylabel('Avg Monthly Return (%)')
for bar, (_, row) in zip(bars, summary.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05 * (1 if bar.get_height() >= 0 else -1),
                 f"{row['Avg Return (%)']:+.2f}%\n(Hit: {row['Hit Rate (%)']:.0f}%)",
                 ha='center', va='bottom' if bar.get_height() >= 0 else 'top',
                 fontsize=8.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig6_regime_analysis.png', bbox_inches='tight')
plt.show()

print('\n--- Regime Summary ---')
print(summary.to_string(index=False))

---
## 7. Summary Table — Latest Snapshot

In [ ]:
latest_date = ecb_df.index.max()
prev_year   = latest_date - pd.DateOffset(months=12)

print('=' * 60)
print(f'  MACRO SNAPSHOT — {latest_date.strftime("%B %Y")}')
print('=' * 60)

# Rates
print('\n  MONETARY POLICY')
for col in ['ECB Deposit Rate (%)', 'Euribor 3M (%)']:
    latest = ecb_df.loc[latest_date, col]
    prev   = ecb_df.loc[ecb_df.index.asof(prev_year), col]
    print(f'  {col:<30} {latest:>6.2f}%   (12M ago: {prev:.2f}%   Δ {latest-prev:+.2f}pp)')

# Inflation
print('\n  INFLATION')
col = 'Eurozone Inflation YoY (%)'
latest = ecb_df.loc[latest_date, col]
prev   = ecb_df.loc[ecb_df.index.asof(prev_year), col]
print(f'  {col:<30} {latest:>6.1f}%   (12M ago: {prev:.1f}%   Δ {latest-prev:+.1f}pp)')

# Equities
print('\n  EQUITY MARKETS (total return last 12M)')
for col in equity_df.columns:
    idx_latest = equity_df.index.asof(latest_date)
    idx_prev   = equity_df.index.asof(prev_year)
    ret = (equity_df.loc[idx_latest, col] / equity_df.loc[idx_prev, col] - 1) * 100
    print(f'  {col:<20}  {ret:+.1f}%')

# Energy
print('\n  ENERGY PRICES (last available)')
for col in energy_df.columns:
    idx_latest = energy_df.index.asof(latest_date)
    idx_prev   = energy_df.index.asof(prev_year)
    latest_val = energy_df.loc[idx_latest, col]
    prev_val   = energy_df.loc[idx_prev,   col]
    chg = (latest_val / prev_val - 1) * 100
    print(f'  {col:<30}  {latest_val:>7.1f}   (YoY: {chg:+.1f}%)')

print('\n' + '=' * 60)

# Save snapshot
snapshot = pd.DataFrame({
    'Indicator': list(ecb_df.columns) + list(equity_df.columns) + list(energy_df.columns),
})
snapshot.to_csv(OUTPUT_DIR / 'macro_snapshot.csv', index=False)

---
## Key Takeaways

- **Monetary policy** transitioned from historic lows (negative DFR) through the fastest hiking cycle in ECB history (2022–23) to a gradual easing phase.
- **Inflation** surged post-COVID, peaked above 10% YoY in late 2022, and has been converging back toward the 2% target.
- **Equity markets** showed strong resilience post-COVID, with a correction during the rate-hiking cycle followed by renewed strength as inflation moderated.
- **Energy prices** were the primary driver of the 2021–22 inflation shock; the gas price spike (+1,200% from pre-crisis) had significant transmission effects on European industrial competitiveness.
- **Regime analysis** suggests European equities historically perform better during cutting cycles (+X%) vs hiking cycles — relevant for PE entry/exit timing decisions.

> This dashboard is designed to provide the macro context required to assess **sector tailwinds/headwinds** for private equity investments, particularly in climate transition and digital infrastructure themes.